# ECG–text likelihood and patch alignment visualizer

This notebook supports both checkpoints produced by **Stage 1 MIMIC report alignment** and **Stage 2 diagnosis-text adaptation**. Configure the paths and candidate strings below. For a downstream checkpoint, leave `candidate_texts = None` to use the diagnosis prompts stored in the checkpoint. For an upstream checkpoint, provide the true report plus one or more comparison reports so the displayed retrieval probabilities are meaningful.

The heatmap is patch–token similarity, not a causal attribution map.


In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision import transforms
from transformers import AutoTokenizer

from filip.model.filip_ecg_model import FILIPECGModel


In [ ]:
# ---- Edit these values ----
checkpoint_path = Path("outputs/filip/mimic_report_pretrain/checkpoints/best.pt")
image_path = Path("data/mimic-iv-ecg/images/EXAMPLE.png")

# Stage 1: supply the matched report and useful distractors.
# Stage 2: set to None to use checkpoint["diagnosis_prompts"].
candidate_texts = [
    "normal sinus rhythm with a normal electrocardiogram",
    "atrial fibrillation with rapid ventricular response",
]
selected_text_index = 0

# Select one or more displayed tokenizer tokens. None uses all content tokens.
selected_token_text = None  # e.g. "fibrillation"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)
config = checkpoint["config"]
# A Stage 1 checkpoint already has this setting. This also makes old Stage 2
# checkpoints explicit when their stored config retained report alignment.
config.setdefault("model", {})["use_report_alignment"] = True
model = FILIPECGModel(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"], strict=False)
model.eval()

if candidate_texts is None:
    candidate_texts = checkpoint.get("diagnosis_prompts")
if not candidate_texts:
    raise ValueError("Provide candidate_texts or use a checkpoint containing diagnosis_prompts")

text_model_name = config["model"].get("text_encoder", config["model"]["vision_encoder"])
tokenizer = AutoTokenizer.from_pretrained(text_model_name)
encoded = tokenizer(
    candidate_texts,
    padding=True,
    truncation=True,
    max_length=config["model"].get("text_max_length", 77),
    return_special_tokens_mask=True,
    return_tensors="pt",
)
content_mask = encoded["attention_mask"].bool() & ~encoded.pop("special_tokens_mask").bool()
prompt_inputs = {
    "input_ids": encoded["input_ids"].to(device),
    "attention_mask": encoded["attention_mask"].to(device),
    "content_mask": content_mask.to(device),
}


In [ ]:
class ExpandToSquare:
    def __init__(self, color=(255, 255, 255)):
        self.color = color
    def __call__(self, image):
        width, height = image.size
        side = max(width, height)
        canvas = Image.new(image.mode, (side, side), self.color)
        canvas.paste(image, ((side - width) // 2, (side - height) // 2))
        return canvas

original_image = Image.open(image_path).convert("RGB")
image_size = config["model"].get("image_size", 224)
transform = transforms.Compose([
    ExpandToSquare(),
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275, 0.40821073],
        std=[0.26862954, 0.26130258, 0.27577711],
    ),
])
image_tensor = transform(original_image).unsqueeze(0).to(device)

with torch.no_grad():
    outputs = model.forward_text_prompts(image_tensor, **prompt_inputs)
logits = outputs["diagnosis_logits"][0]
similarities = outputs["patch_prompt_similarity"][0]  # [C, P, T]

is_downstream = (checkpoint.get("diagnosis_mode") == "text_prompts" or
                 config.get("model", {}).get("diagnosis_mode") == "text_prompts")
likelihoods = torch.sigmoid(logits) if is_downstream else torch.softmax(logits, dim=0)
for text, logit, probability in zip(candidate_texts, logits.cpu(), likelihoods.cpu()):
    label = "multilabel likelihood" if is_downstream else "candidate retrieval probability"
    print(f"{probability.item():.4f} ({label}; logit={logit.item():.3f})  {text}")


In [ ]:
token_ids = prompt_inputs["input_ids"][selected_text_index].cpu().tolist()
tokens = tokenizer.convert_ids_to_tokens(token_ids)
valid = content_mask[selected_text_index]
valid_indices = valid.nonzero(as_tuple=False).flatten().tolist()
print("Content tokens:", [(index, tokens[index]) for index in valid_indices])

if selected_token_text:
    needle = selected_token_text.lower().replace(" ", "")
    selected_indices = [
        index for index in valid_indices
        if needle in tokens[index].lower().replace("</w>", "").replace(" ", "")
    ]
    if not selected_indices:
        raise ValueError(f"No content token matched {selected_token_text!r}; inspect Content tokens above")
else:
    selected_indices = valid_indices

# Mean aggregation gives word/phrase/report-level grounding.
patch_scores = similarities[selected_text_index, :, selected_indices].mean(dim=-1).cpu().numpy()
patch_size = config["model"].get("patch_size", 32)
grid_size = image_size // patch_size
if patch_scores.size != grid_size * grid_size:
    raise ValueError(f"Expected {grid_size ** 2} patches, received {patch_scores.size}")
heatmap = patch_scores.reshape(grid_size, grid_size)
heatmap = (heatmap - heatmap.min()) / max(float(heatmap.max() - heatmap.min()), 1e-8)
heatmap_image = Image.fromarray(np.uint8(heatmap * 255)).resize(original_image.size, Image.Resampling.BICUBIC)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].imshow(original_image)
axes[0].set_title("ECG image")
axes[0].axis("off")
axes[1].imshow(original_image)
axes[1].imshow(np.asarray(heatmap_image), cmap="jet", alpha=0.35, vmin=0, vmax=255)
selection = selected_token_text or "all content tokens"
axes[1].set_title(f"Patch alignment: {selection}\n{candidate_texts[selected_text_index]}")
axes[1].axis("off")
plt.tight_layout()
plt.show()


## Reading the output

* **Stage 1:** probabilities are relative retrieval probabilities across the candidate texts supplied in the setup cell. Include distractors; a single candidate always has probability 1.
* **Stage 2 text adaptation:** values are independent sigmoid likelihoods because diagnoses are multilabel. Use validation-tuned per-class thresholds for final decisions.
* Set `selected_token_text` to inspect a word token. Leave it as `None` to average all content-token maps for the selected report or diagnosis prompt.
